# Court Keypoint Model - YOLOv8 Pose Training

Trains a YOLOv8-pose model on court keypoint annotations (8 points).

In [ ]:
# @title 1. Install & Mount Drive
!pip install -q ultralytics

from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import os
print("Ready.")

In [ ]:
# @title 2. Config
DATASET_PATH = "/content/drive/MyDrive/annotation/court_dataset"  # @param {type:"string"}
EPOCHS = 100  # @param {type:"integer"}
BATCH_SIZE = 16  # @param {type:"integer"}
IMGSZ = 640  # @param {type:"integer"}
PATIENCE = 20  # @param {type:"integer"} early stopping
MODEL = "yolov8n-pose.pt"  # @param ["yolov8n-pose.pt", "yolov8s-pose.pt", "yolov8m-pose.pt"]

assert os.path.exists(DATASET_PATH), f"Dataset not found: {DATASET_PATH}"
assert os.path.exists(f"{DATASET_PATH}/dataset.yaml"), "dataset.yaml not found"
print(f"Dataset: {DATASET_PATH}")
print(f"Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, Img: {IMGSZ}")

In [ ]:
# @title 3. Train
model = YOLO(MODEL)

results = model.train(
    data=f"{DATASET_PATH}/dataset.yaml",
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    patience=PATIENCE,
    project="runs/pose",
    name="train",
)

print("Training done.")

In [ ]:
# @title 4. Validate
best_weights = "runs/pose/train/weights/best.pt"
model = YOLO(best_weights)

metrics = model.val(data=f"{DATASET_PATH}/dataset.yaml")

print("\n--- Metrics ---")
if hasattr(metrics, 'box'):
    print(f"Box mAP50: {getattr(metrics.box.map50, 'item', lambda: metrics.box.map50)() if hasattr(metrics.box, 'map50') else 'N/A'}")
if hasattr(metrics, 'pose'):
    print(f"Pose mAP50: {getattr(metrics.pose.map50, 'item', lambda: metrics.pose.map50)() if hasattr(metrics.pose, 'map50') else 'N/A'}")
print(metrics)

In [ ]:
# @title 5. Test on Sample Images
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

model = YOLO("runs/pose/train/weights/best.pt")
val_dir = Path(f"{DATASET_PATH}/images/val")
imgs = list(val_dir.glob("*.jpg"))[:6]

if not imgs:
    print("No val images found.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(14, 10))
    axes = axes.flatten()
    for i, p in enumerate(imgs):
        r = model.predict(str(p), verbose=False)[0]
        plot = r.plot()  # BGR
        axes[i].imshow(cv2.cvtColor(plot, cv2.COLOR_BGR2RGB))
        axes[i].set_title(p.name)
        axes[i].axis("off")
    for j in range(i + 1, 6):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# @title 6. Save Model to Drive
import shutil

DRIVE_MODEL_DIR = "/content/drive/MyDrive/annotation/court_model"  # @param {type:"string"}
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

shutil.copy("runs/pose/train/weights/best.pt", f"{DRIVE_MODEL_DIR}/best.pt")
shutil.copy("runs/pose/train/weights/last.pt", f"{DRIVE_MODEL_DIR}/last.pt")

print(f"Saved best.pt and last.pt to {DRIVE_MODEL_DIR}")